In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
class AgentState(TypedDict):
    query: str
    category: str
    response: str

def classify(state):
    query = state["query"].lower()

    if any(word in query for word in ["payment", "refund", "money"]):
        category = "human_review"
    else:
        category = "general"

    return {"category": category}

def route(state):
    return {"category": state["category"]}

def respond(state):
    if state["category"] == "human_review":
        feedback = interrupt("Human approval required")
        return {"response": f"Human decision: {feedback}"}

    return {"response": "General query processed successfully"}

builder = StateGraph(AgentState)

builder.add_node("classify", classify)
builder.add_node("route", route)
builder.add_node("respond", respond)

builder.add_edge(START, "classify")
builder.add_edge("classify", "route")
builder.add_edge("route", "respond")
builder.add_edge("respond", END)

graph = builder.compile(checkpointer=MemorySaver())

print("LangGraph workflow created successfully!")

LangGraph workflow created successfully!


In [3]:
queries = [
    "What is artificial intelligence?",
    "Explain machine learning",
    "I need a refund",
    "What is Python?",
    "My payment failed"
]

for i, query in enumerate(queries):
    config = {"configurable": {"thread_id": str(i)}}
    result = graph.invoke({"query": query}, config)

    print("Query:", query)
    print("Category:", result["category"])
    print("Response:", result.get("response", "Waiting for human approval"))
    print("-" * 40)

Query: What is artificial intelligence?
Category: general
Response: General query processed successfully
----------------------------------------
Query: Explain machine learning
Category: general
Response: General query processed successfully
----------------------------------------
Query: I need a refund
Category: human_review
Response: Waiting for human approval
----------------------------------------
Query: What is Python?
Category: general
Response: General query processed successfully
----------------------------------------
Query: My payment failed
Category: human_review
Response: Waiting for human approval
----------------------------------------


In [4]:
config = {"configurable": {"thread_id": "2"}}

result = graph.invoke(
    Command(resume="Refund approved"),
    config
)

print("Final response:", result["response"])


Final response: Human decision: Refund approved


In [5]:
builder = StateGraph(AgentState)

builder.add_node("classify", classify)
builder.add_node("route", route)
builder.add_node("respond", respond)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    lambda state: state["category"],
    {"general": "respond", "human_review": "route"}
)
builder.add_edge("route", "respond")
builder.add_edge("respond", END)

graph = builder.compile(checkpointer=MemorySaver())

print("Conditional routing configured successfully!")

Conditional routing configured successfully!


In [6]:
for i, query in enumerate(queries):
    config = {"configurable": {"thread_id": f"final-{i}"}}
    result = graph.invoke({"query": query}, config)

    if result.get("__interrupt__"):
        result = graph.invoke(Command(resume="Approved"), config)

    print(query, "→", result["category"], "→", result["response"])

What is artificial intelligence? → general → General query processed successfully
Explain machine learning → general → General query processed successfully
I need a refund → human_review → Human decision: Approved
What is Python? → general → General query processed successfully
My payment failed → human_review → Human decision: Approved


## Test Results

- Three-node LangGraph workflow: Passed
- Five input tests: Passed
- Conditional routing: Passed
- Human approval interruption: Passed
- Resume after approval: Passed

## Conclusion

Successfully implemented a LangGraph workflow with
conditional routing and human-in-the-loop approval.